# Medical Data Pipeline - Google Colab

This notebook runs the complete medical data pipeline for demonstration purposes.

**Pipeline Structure:**
1. Setup environment
2. Upload 1-year sample data
3. Run parsing scripts (PDF/TXT → CSV)
4. Run statistical analysis
5. Generate PDF report
6. View results

**Note:** Only use the provided 1-year anonymized sample data.

In [ ]:
# Cell 1: Setup Environment
print("="*60)
print("STEP 1: SETTING UP ENVIRONMENT")
print("="*60)

!python /content/medical-pipeline/setup_colab.py

## Upload Data Files

**Using the file browser on the left (📁 icon):**

1. **Ritter PDF files** to:
   - `/content/medical-pipeline/data/RitterPDF/Ritter1/`
   - `/content/medical-pipeline/data/RitterPDF/Ritter2/`

2. **Statim text files** to:
   - `/content/medical-pipeline/data/Statim/StatimA/`
   - `/content/medical-pipeline/data/Statim/StatimB/`

**Important:** Only upload the 1-year sample data provided in the repository.

In [ ]:
# Cell 2: Verify Uploaded Data
print("="*60)
print("STEP 2: VERIFYING UPLOADED DATA")
print("="*60)

import sys
sys.path.insert(0, '/content/medical-pipeline/config')
import paths

def check_data_files(directory, extension):
    """Check files in directory"""
    from pathlib import Path
    files = list(Path(directory).glob(f"*{extension}"))
    print(f"{directory.name}/:")
    if files:
        for f in files[:3]:  # Show first 3 files
            size_mb = f.stat().st_size / (1024*1024)
            print(f"  • {f.name} ({size_mb:.2f} MB)")
        if len(files) > 3:
            print(f"  • ... and {len(files)-3} more files")
    else:
        print(f"  ⚠ No {extension} files found")
    print()

print("Checking data directories:")
check_data_files(paths.RITTER1_PDF_DIR, ".pdf")
check_data_files(paths.RITTER2_PDF_DIR, ".pdf")
check_data_files(paths.STATIMA_DIR, ".txt")
check_data_files(paths.STATIMB_DIR, ".txt")

In [ ]:
# Cell 3: Run Parsing Pipeline
print("="*60)
print("STEP 3: RUNNING PARSING PIPELINE")
print("="*60)

# Run the main pipeline script
!cd /content/medical-pipeline && python run_pipeline.py

In [ ]:
# Cell 4: View Parsing Results
print("="*60)
print("STEP 4: PARSING RESULTS")
print("="*60)

import pandas as pd
from pathlib import Path

print("Generated CSV files:")
csv_files = list(paths.PARSING_RESULTS_DIR.glob("*.csv"))
for csv_file in csv_files:
    try:
        df = pd.read_csv(csv_file)
        print(f"\n{csv_file.name}:")
        print(f"  Rows: {len(df):,}")
        print(f"  Columns: {len(df.columns)}")
        print(f"  Columns: {', '.join(df.columns[:5])}" + ("..." if len(df.columns) > 5 else ""))
        print(f"  Sample data:")
        print(df.head(2).to_string(index=False))
    except Exception as e:
        print(f"\nError reading {csv_file.name}: {e}")

In [ ]:
# Cell 5: Run Statistical Analysis
print("="*60)
print("STEP 5: RUNNING STATISTICAL ANALYSIS")
print("="*60)

# Import and run merged2.py
sys.path.insert(0, '/content/medical-pipeline/scripts/statistics_codes')
print("Running merged2.py...")
!cd /content/medical-pipeline && python scripts/statistics_codes/merged2.py

In [ ]:
# Cell 6: Generate PDF Report
print("="*60)
print("STEP 6: GENERATING PDF REPORT")
print("="*60)

# Run report2.py
print("Running report2.py...")
!cd /content/medical-pipeline && python scripts/statistics_codes/report2.py

# Check for generated report
pdf_reports = list(paths.RESULTS_DIR.rglob("*.pdf"))
if pdf_reports:
    print("\nGenerated PDF reports:")
    for pdf in pdf_reports:
        size_mb = pdf.stat().st_size / (1024*1024)
        print(f"  • {pdf.name} ({size_mb:.2f} MB)")
else:
    print("\n⚠ No PDF reports found")

In [ ]:
# Cell 7: View All Results
print("="*60)
print("STEP 7: COMPLETE RESULTS SUMMARY")
print("="*60)

from pathlib import Path
import os

def list_result_files(directory, indent=0):
    """Recursively list result files"""
    prefix = "  " * indent
    try:
        items = sorted(os.listdir(directory))
        for item in items:
            item_path = os.path.join(directory, item)
            if os.path.isdir(item_path):
                print(f"{prefix}📁 {item}/")
                list_result_files(item_path, indent + 1)
            else:
                size = os.path.getsize(item_path) / 1024  # KB
                print(f"{prefix}  📄 {item} ({size:.1f} KB)")
    except Exception as e:
        print(f"{prefix}⚠ Error: {e}")

print("\nComplete results structure:")
list_result_files(str(paths.RESULTS_DIR))

# Display a sample visualization if available
visual_files = list(paths.VISUAL_RESULTS_DIR.glob("*.png")) + list(paths.VISUAL_RESULTS_DIR.glob("*.jpg"))
if visual_files:
    print("\nSample visualizations:")
    for img in visual_files[:2]:  # Show first 2
        print(f"  • {img.name}")
    
    # Display first image
    try:
        from IPython.display import Image, display
        display(Image(str(visual_files[0])))
    except:
        pass